In [24]:
from azure.identity import DefaultAzureCredential
from azure.ai.ml import MLClient , command , Input , Output
from azure.ai.ml.entities import Data , Environment
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import ComputeInstance
from azure.ai.ml.entities import AmlCompute

from datetime import datetime
from zoneinfo import ZoneInfo


from azure.ai.ml.dsl import pipeline


In [25]:
from azure.ai.ml import MLClient
from azure.ai.ml.dsl import pipeline
from azure.ai.ml.entities import (
    BatchEndpoint,
    BatchDeployment,
    BatchRetrySettings
)
from azure.identity import DefaultAzureCredential


In [2]:

# Connect mlClient
credential = DefaultAzureCredential()

ml_client = MLClient.from_config(
    credential=DefaultAzureCredential()
)


Found the config file in: /config.json
Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


In [3]:

ml_client = MLClient(
    credential=credential,
    subscription_id = ml_client.subscription_id,
    resource_group_name = ml_client.resource_group_name,
    workspace_name = ml_client.workspace_name
)

Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


In [4]:
env_name = "Pipeline_Flight-env"
conda_file = "Environment.yml"
image_details ="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest"
DataStore_name = 'flightsdatastrore'
Blob_input = 'flightdelaydata.csv'

compute_instance_name = 'CPU432'

Prepare_data = "Prepare_date.py"
Train_data = 'Train_data.py'

In [5]:

# get data set
datastore = ml_client.datastores.get(DataStore_name)
print(datastore)
path = f"azureml://subscriptions/{ml_client.subscription_id}/resourcegroups/{ml_client.resource_group_name}/workspaces/{ml_client.workspace_name}/datastores/{datastore.name}/paths/{Blob_input}"


account_name: dp100n
container_name: flightcontainer
credentials: {}
endpoint: core.windows.net
id: /subscriptions/49297e8f-2e51-410c-aee9-69db39628913/resourceGroups/dp-100_trail/providers/Microsoft.MachineLearningServices/workspaces/dp-100/datastores/flightsdatastrore
name: flightsdatastrore
protocol: https
tags: {}
type: azure_blob



In [6]:

# Create environment
env = Environment(
    name = f"{env_name}",
    description="Custom AML environment",
    conda_file=f"{conda_file}",
    image = f"{image_details}" 
    # version = '1'
)

In [7]:


# Register environment
ml_client.environments.create_or_update(env)

# print("Environment created successfully")

Environment({'arm_type': 'environment_version', 'latest_version': None, 'image': 'mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest', 'intellectual_property': None, 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'Pipeline_Flight-env', 'description': 'Custom AML environment', 'tags': {}, 'properties': {'azureml.labels': 'latest'}, 'print_as_yaml': False, 'id': '/subscriptions/49297e8f-2e51-410c-aee9-69db39628913/resourceGroups/dp-100_trail/providers/Microsoft.MachineLearningServices/workspaces/dp-100/environments/Pipeline_Flight-env/versions/1', 'Resource__source_path': '', 'base_path': '/mnt/batch/tasks/shared/LS_root/mounts/clusters/cpu432/code/Users/brenjoelsikha/PPipeline_Project', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x74fc951ad810>, 'serialize': <msrest.serialization.Serializer object at 0x74fc951af340>, 'version': '1', 'conda_file': {'channels': ['conda-forge'], 'dependencies': ['python=3

In [8]:




# # Register the original data set  URI FILE
# raw_data = Data(
#     path=path,
#     type=AssetTypes.URI_FILE,
#     name=DataAsset_Name,
#     # version="Main",
#     description="Raw flight delay dataset"
# )

# ml_client.data.create_or_update(raw_data)

In [9]:
# # Create a compute instance

# compute_instance_name = "test-my-compute-instance"
# compute_instance = ComputeInstance(
#     name=compute_instance_name ,
#     size="STANDARD_DS3_V2",
# )

# ml_client.begin_create_or_update(compute_instance).result()

# print("Compute instance created")

In [10]:

# # Create compute cluster 
# compute_cluster_name = 'testCluster'
# cpu_cluster = AmlCompute(
#     name=compute_cluster_name,
#     type="amlcompute",
#     size="STANDARD_DS2_V2",
#     min_instances=0,
#     max_instances=1,
#     idle_time_before_scale_down=120,
# )

# ml_client.begin_create_or_update(cpu_cluster).result()

# print("Compute cluster created")

In [11]:


# Step 1 - Prepare Data
prepare_data_step = command(
    name= 'prepare_data',
    display_name="Prepare Data",

    code="./src",

    command=f"""
    python {Prepare_data} \
    --input_data ${{{{inputs.input_data}}}} \
    --prepped_data ${{{{outputs.prepped_data}}}}
    """,

    inputs={
        "input_data": Input(
            type=AssetTypes.URI_FILE,
            path=path
        )
    },

    outputs={
        "prepped_data": Output(
            type=AssetTypes.URI_FOLDER
        )
    },

    environment=f'{env_name}@latest',
    compute=compute_instance_name
)

# Step 2 - Train Model
train_model_step = command(
    name='train_data',
    display_name="Train Model",

    code="./src",

    command=f"""
    python {Train_data} \
    --prepped_data ${{{{inputs.prepped_data}}}}
    """,

    inputs={
        "prepped_data": Input(
            type=AssetTypes.URI_FOLDER
        )
    },

    environment=f'{env_name}@latest',  # "pipeline_env@latest",
    compute=compute_instance_name
)


In [12]:

# Build Pipeline
@pipeline()
def training_pipeline():

    # Step 1 execution
    prep_step = prepare_data_step()
    prep_step.compute=compute_instance_name
    # Step 2 execution with output from step 1
    train_step = train_model_step(
        prepped_data=prep_step.outputs.prepped_data
    )
    train_step.compute = compute_instance_name
    return {
        "prepared_data": prep_step.outputs.prepped_data
    }


In [13]:

# Create pipeline job
pipeline_job = training_pipeline()

# Equivalent of regenerate_outputs=True
pipeline_job.settings.force_rerun = True

# Submit pipeline
returned_job = ml_client.jobs.create_or_update(pipeline_job)

ml_client.jobs.stream(returned_job.name)

Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
pathOnCompute is not a known attribute

RunId: patient_lime_kdv3ph2jr1
Web View: https://ml.azure.com/runs/patient_lime_kdv3ph2jr1?wsid=/subscriptions/49297e8f-2e51-410c-aee9-69db39628913/resourcegroups/dp-100_trail/workspaces/dp-100

Streaming logs/azureml/executionlogs.txt

[2026-05-28 08:47:11Z] Submitting 1 runs, first five are: f62258a7:f4918aeb-ec2e-4511-b1c8-da9a2f7027d8
[2026-05-28 08:49:13Z] Completing processing run id f4918aeb-ec2e-4511-b1c8-da9a2f7027d8.
[2026-05-28 08:49:13Z] Submitting 1 runs, first five are: 643c1cd5:6a1bee43-ebcd-4345-9935-cacfd8c45ae8
[2026-05-28 08:49:47Z] Completing processing run id 6a1bee43-ebcd-4345-9935-cacfd8c45ae8.

Execution Summary
RunId: patient_lime_kdv3ph2jr1
Web View: https://ml.azure.com/runs/patient_lime_kdv3ph2jr1?wsid=/subscriptions/49297e8f-2e51-410c-aee9-69db39628913/resourcegroups/dp-100_trail/workspaces/dp-100



In [14]:
# # # Stop compute instance
# ml_client.compute.begin_stop(compute_instance_name).wait()

# print("Stopped")

In [15]:
# # # Delete compute instance
# ml_client.compute.begin_delete(compute_instance_name).wait()

# print("Compute instance deleted")

In [16]:
# Delete Compute Cluster 
# ml_client.compute.begin_delete(compute_cluster_name).wait()

In [17]:
from azure.identity import InteractiveBrowserCredential

auth_credential = InteractiveBrowserCredential()

token = credential.get_token(
    "https://ml.azure.com/.default"
)

access_token = token.token

In [18]:
import requests
import json

exp_name = "pipeline_endpoint_exp"
endpoint = "https://eastus2.api.azureml.ms/pipelines/v1.0/subscriptions/49297e8f-2e51-410c-aee9-69db39628913/resourceGroups/dp-100_trail/providers/Microsoft.MachineLearningServices/workspaces/dp-100/PipelineRuns/PipelineEndpointSubmit/Id/6bb32fbc-2a8f-4c1e-8464-24bc50f00171"

headers = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json"
}

payload = {
    "ExperimentName": "PipelineTrigger"
}

response = requests.post(
    endpoint,
    headers=headers,
    json=payload
)

print(response.status_code)
print(response.text)

200
{
  "Description": null,
  "Status": {
    "StatusCode": 0,
    "StatusDetail": null,
    "CreationTime": "2026-05-28T08:50:08.9510443Z",
    "EndTime": null
  },
  "GraphId": "379e15eb-c7b6-40ae-8078-80a0a2a71dc7",
  "IsSubmitted": false,
  "HasErrors": false,
  "HasWarnings": false,
  "UploadState": 0,
  "ParameterAssignments": {},
  "DataPathAssignments": {},
  "DataSetDefinitionValueAssignments": {},
  "AssetOutputSettingsAssignments": {
    "prepared_data": {
      "Path": "azureml://datastores/${{default_datastore}}/paths/azureml/${{name}}/${{output_name}}/",
      "Type": 1,
      "Options": null,
      "DataStoreMode": 1,
      "Name": null,
      "Version": null
    }
  },
  "RunHistoryExperimentName": "PipelineTrigger",
  "DisplayName": null,
  "PipelineRunId": "0c8d5b3e-284f-4b08-a223-2b612cf3d6c1",
  "PipelineId": "26736f99-7f19-402f-84bc-d3fb6f1eed5c",
  "PipelineEndpointId": "6bb32fbc-2a8f-4c1e-8464-24bc50f00171",
  "RunSource": "Unavailable",
  "RunType": 0,
  "Total

## 2


## 3

In [32]:
# ---------------------------------------------------
# CREATE PIPELINE JOB
# ---------------------------------------------------

pipeline_job = training_pipeline()

pipeline_job.settings.force_rerun = True


# ---------------------------------------------------
# CREATE BATCH ENDPOINT
# ---------------------------------------------------

from azure.ai.ml.entities import (
    BatchEndpoint,
    PipelineComponentBatchDeployment
)

endpoint_name = "training-pipeline-endpoint"

endpoint = BatchEndpoint(
    name=endpoint_name,
    description="Training pipeline endpoint"
)

ml_client.batch_endpoints.begin_create_or_update(endpoint).result()

print("Endpoint created")


# ---------------------------------------------------
# CREATE DEPLOYMENT
# ---------------------------------------------------

deployment = PipelineComponentBatchDeployment(
    name="training-deployment",
    endpoint_name=endpoint_name,

    component=pipeline_job.component
)

ml_client.batch_deployments.begin_create_or_update(deployment).result()

print("Deployment created")


# ---------------------------------------------------
# SET DEFAULT DEPLOYMENT
# ---------------------------------------------------

endpoint = ml_client.batch_endpoints.get(endpoint_name)

endpoint.defaults.deployment_name = "training-deployment"

ml_client.batch_endpoints.begin_create_or_update(endpoint).result()

print("Default deployment set")


# ---------------------------------------------------
# INVOKE ENDPOINT
# ---------------------------------------------------

job = ml_client.batch_endpoints.invoke(
    endpoint_name=endpoint_name
)

print(job)
ml_client.jobs.stream(
    "pipelinejob-e71fd4a1-7ee1-4053-8afd-8a29f3a34f56"
)

Endpoint created
Deployment created
Default deployment set
{'additional_properties': {}, 'id': '/subscriptions/49297e8f-2e51-410c-aee9-69db39628913/resourceGroups/dp-100_trail/providers/Microsoft.MachineLearningServices/workspaces/dp-100/batchEndpoints//jobs/pipelinejob-e71fd4a1-7ee1-4053-8afd-8a29f3a34f56', 'name': 'pipelinejob-e71fd4a1-7ee1-4053-8afd-8a29f3a34f56', 'type': 'Microsoft.MachineLearningServices/workspaces/batchEndpoints/jobs', 'properties': <azure.ai.ml._restclient.v2020_09_01_dataplanepreview.models._models_py3.BatchJob object at 0x74fc3abfe380>, 'system_data': <azure.ai.ml._restclient.v2020_09_01_dataplanepreview.models._models_py3.SystemData object at 0x74fc3abfe7a0>}


In [33]:
ml_client.jobs.stream(
    "pipelinejob-e71fd4a1-7ee1-4053-8afd-8a29f3a34f56"
)

RunId: pipelinejob-e71fd4a1-7ee1-4053-8afd-8a29f3a34f56
Web View: https://ml.azure.com/runs/pipelinejob-e71fd4a1-7ee1-4053-8afd-8a29f3a34f56?wsid=/subscriptions/49297e8f-2e51-410c-aee9-69db39628913/resourcegroups/dp-100_trail/workspaces/dp-100

Execution Summary
RunId: pipelinejob-e71fd4a1-7ee1-4053-8afd-8a29f3a34f56
Web View: https://ml.azure.com/runs/pipelinejob-e71fd4a1-7ee1-4053-8afd-8a29f3a34f56?wsid=/subscriptions/49297e8f-2e51-410c-aee9-69db39628913/resourcegroups/dp-100_trail/workspaces/dp-100



## 1